Goal: To build a binary text classifier that identifies subjective media bias. 

This tool should help public by highlighting biased language in news media.

Intermediate objective: Detect media bias using Logistic Regression for binary classification.

Binary prediction: Biased(1) vs. Neutral(0)

Dataset:  newsmediabias/debiased_dataset (streamed from HuggingFace)

In [36]:
# About dataset
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

print("Loading dataset...")
dataset = load_dataset("newsmediabias/debiased_dataset", split="train")
print(dataset)

df_original = dataset.to_pandas()

print(df_original.loc[2, 'biased_text'])  # Note: .loc to access a specific row and column
print(df_original.loc[2, 'aspect'])
print(df_original.loc[2, 'debiased_text'])

# print(df_original.head(1))

df_original['biased_text'].to_csv('biased_text_sample.csv', index=False)



Loading dataset...


Using the latest cached version of the dataset since newsmediabias/debiased_dataset couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /Users/sneharoy/.cache/huggingface/datasets/newsmediabias___debiased_dataset/default/0.0.0/bb97995835c1c645d1d0ffef39bcb7101b3aef9b (last modified on Thu Mar 12 08:18:45 2026).


Dataset({
    features: ['biased_text', 'dimension', 'aspect', 'biased_profain_words', 'bias_label', 'debiased_text'],
    num_rows: 7492
})
This case is more than about rape , which in itself is despicable : it 's also about Cosby 's abuse of his celebrity status and his total lack of shame for his actions . From a legal standpoint , did he really perpetrate all the things for which he is accused or is he `` only '' guilty of a few ? To me that does n't matter : the fact that he willingly admits to giving three rape pills to a woman tells me enough about Cosby 's character . Vile and disgusting is what he is to me , never mind whether or not he walks a free man . Bill Cosby is the epitomy of hypocrisy , narcissism and psychopathic behaviour . I have zero respect for him .
Criminal record discrimination
This case is more than about sexual assault, which is morally wrong: it is also about Cosby's misuse of his fame and his lack of remorse for his actions. From a legal perspective, wheth

In [28]:
# Design the dataset for binary classification: 1 for biased, 0 for neutral/unbiased
df_biased = pd.DataFrame({'text': df_original['biased_text'], 'label': 1})
df_unbiased = pd.DataFrame({'text': df_original['debiased_text'], 'label': 0})

# Shuffle to prevent skewed learning w/ .concat and .sample
df = pd.concat([df_biased, df_unbiased]).sample(frac=1, random_state=42).reset_index(drop=True)
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], 
    df['label'], 
    train_size=0.8,
    test_size=0.2,
    random_state=42)

print(f"Total sentences: {len(df)}")
print(f"df_biased rows: {len(df_biased)}")
print(f"df_unbiased rows: {len(df_unbiased)}")
# display(df.head(3))

Total sentences: 14984
df_biased rows: 7492
df_unbiased rows: 7492


df_biased and df_unbiased are 2D matrices of equal n x m

It's pulling the biased_text column out as one dataset (labeled 1) and the debiased_text column out as another (labeled 0)

In [63]:
# Tokenizing & Vectorizing

import re
import sys
import os
from sklearn.feature_extraction.text import CountVectorizer


sys.path.append(os.path.abspath("../src"))
import contractions

def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # "do n't" -> "don't", "it 's" -> "it's"
    text = re.sub(r"\s+n't\b", "n't", text)     
    text = re.sub(r"\s+'(s|ve|ll|d|re|m)\b", r"'\1", text)  
    
    for word, expanded in contractions.CONTRACTION_MAP.items():
        text = re.sub(r'\b' + re.escape(word) + r'\b', expanded, text, flags=re.IGNORECASE)
    
    # Strip HTML tags 
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'&\s*#\s*\d+\s*;', ' ', text)
    
    # Strip urls
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    
    # Strip Twitter/X handles
    text = re.sub(r'@\s*\w+', ' ', text)
    text = re.sub(r'\bRT\b', ' ', text)
    
    # Repeated quotes/backticks
    text = re.sub(r"[`'\"]{2,}", ' ', text)

    # Strip whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text.lower()

# CountVectorizer for Unigrams and Bigrams
vectorizer = CountVectorizer(
    preprocessor=clean_text, 
    ngram_range=(1, 2),
    max_features=10000,
    min_df=3,
    token_pattern=r'\b[a-z]{2,}\b'
)
print("Raw text to vector:")
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Extracted {X_train_vec} unique unigram and bigram features.")

Raw text to vector:
Extracted <Compressed Sparse Row sparse matrix of dtype 'int64'
	with 571314 stored elements and shape (11987, 10000)>
  Coords	Values
  (0, 9902)	1
  (0, 651)	1
  (0, 7522)	1
  (0, 9971)	1
  (0, 5401)	1
  (0, 3141)	2
  (0, 4487)	1
  (0, 7824)	2
  (0, 4216)	3
  (0, 2776)	1
  (0, 9204)	1
  (0, 7277)	1
  (0, 3538)	1
  (0, 3437)	2
  (0, 5409)	1
  (0, 6308)	1
  (0, 4710)	1
  (0, 7654)	1
  (0, 5842)	1
  (0, 9895)	1
  (0, 1291)	1
  (0, 4619)	1
  (0, 5578)	4
  (0, 2672)	1
  (0, 5358)	1
  :	:
  (11986, 8296)	1
  (11986, 9645)	1
  (11986, 3398)	1
  (11986, 745)	1
  (11986, 1949)	3
  (11986, 7890)	1
  (11986, 6892)	1
  (11986, 9076)	1
  (11986, 9078)	1
  (11986, 3432)	1
  (11986, 3395)	1
  (11986, 8932)	1
  (11986, 3392)	1
  (11986, 7658)	1
  (11986, 7817)	1
  (11986, 8015)	2
  (11986, 5140)	1
  (11986, 7341)	1
  (11986, 1615)	1
  (11986, 6618)	1
  (11986, 9626)	1
  (11986, 9456)	1
  (11986, 5229)	1
  (11986, 8589)	1
  (11986, 3393)	1 unique unigram and bigram features.


X_train_vec is a sparse matrix of [11,987 x 10,000] with unigrams, bigrams, 80% training size hyperparameters. 

X_test_vec is a sparse matrix of [2,997 x 10,000] with unigrams, bigrams, 20% training size hyperparameters.

The 10k features for test are mapped into that same 10,000-column space learned during training. 

We ignore tokens that are digits, single char., and that appear in <3 documents. 

Next: Train LR for this classification <- since the input is very sparse yet high-dem. 

PROBLEM: For the sake of trying to keep bias indicating words like "no", "we/us" The model will pick up words like "is", "ok", "my," Which is already plenty of noise. The model is about to be trained on some garbage.

In [66]:
print(vectorizer.get_feature_names_out()[7824])
print(vectorizer.get_feature_names_out()[4216])

that
is


In [ ]:
# Training 

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, log_loss

# Minimize Cross-Entropy loss via gradient descent
print("Training the Logistic Regression model...")
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)
print("Training is done!")

y_pred = model.predict(X_test_vec)             
y_pred_proba = model.predict_proba(X_test_vec) 


print("Model performance:")
print(f"Accuracy:                   {accuracy_score(y_test, y_pred):.4f}")
print(f"Cross-Entropy Loss:         {log_loss(y_test, y_pred_proba):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Training the Logistic Regression model...
Training is done!
Model performance:
Accuracy:                   0.8338
Cross-Entropy Loss:         0.3692

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.82      0.83      1473
           1       0.83      0.85      0.84      1524

    accuracy                           0.83      2997
   macro avg       0.83      0.83      0.83      2997
weighted avg       0.83      0.83      0.83      2997



The model performs consistently across both classes, with no significant gap b/w predicting one label over the other. 

An 83% accuracy with typical bag-of-words approaches. The near-equal precision and recall indicate the model neither over-flags nor under-flags biased content.

Still, not a good model given the preprocessing that can cap the performance and BoW ignores word order beyond bigrams and lacks semantic understanding.

In [77]:
import numpy as np

feature_names = np.array(vectorizer.get_feature_names_out())
weights = model.coef_[0]
weight_df = pd.DataFrame({'Feature': feature_names, 'Weight': weights})

print("Top words that indicate BIASED text (Class 1):")
display(weight_df.sort_values(by='Weight', ascending=False).head(10))

print("Top words that indicate NEUTRAL text (Class 0):")
display(weight_df.sort_values(by='Weight', ascending=True).head(10))

Top words that indicate BIASED text (Class 1):


,Feature,Weight
3745,http,2.533885
362,amp,2.100371
7586,stupid,2.006828
3829,im,1.961729
7613,sucks,1.938090
3458,hate,1.840120
1395,can not,1.756491
2605,error,1.617991
7212,shit,1.544337
2366,dont,1.528296


Top words that indicate NEUTRAL text (Class 0):


,Feature,Weight
4039,individuals,-2.909041
9123,uncomfortable,-2.580574
8982,toremove,-2.507874
9155,unfortunate,-2.099967
1503,challenging,-2.045922
2563,enjoyable,-1.985377
2560,enjoy,-1.968810
7259,significant,-1.896484
2647,even though,-1.672331
8994,towards,-1.625465


Looks like the model learned to push the positive weights towards class 1 (biased), negative toward class 0 (neutral).

Preprocessing is stil exposing junk (http, amp, im)

Still picking expected words (stupid, sucks, hate, shit) 

On the neutral side, some things don't make sense like "toremove" likely "to remove" mashed together. 

Still unclear on why some unigrams are rather assigned positive weights than negative.

In [76]:
print(df[df['label'] == 1]['text'].str.contains('lies').sum()) # Freq. in Biased
print(df[df['label'] == 0]['text'].str.contains('lies').sum()) # Freq. in Neutral

177
184


"Lies" appear nearly identical times and not assigned any significant weight in any direction So it's a good sign of the model working, as it's not based by how much both classes have in common

In [78]:
# Test the model on some new sentences
test_sentence =  "The article was published on May 10th."

sentence_vec = vectorizer.transform([test_sentence])

prediction = model.predict(sentence_vec)[0]
# print(f"The text is (0 = Neutral, 1 = Biased): {prediction}")

# Confidence for each class (0=Neutral, 1=Biased)
class_confidence = model.predict_proba(sentence_vec).flatten()

print(f"Input Text: '{test_sentence}'\n")

if prediction == 1:
    print(f"Verdict: 🚨 BIASED")
    print(f"Confidence: {class_confidence[1] * 100:.2f}%")
else:
    print(f"Verdict: ✅ NEUTRAL")
    print(f"Confidence: {class_confidence[0] * 100:.2f}%")

Input Text: 'The article was published on May 10th.'

Verdict: 🚨 BIASED
Confidence: 56.41%


Future work:
- Try more complex models (Neural Networks, Transformers like BERT)
- Use dense vectors for more MEANINGS (TF-IDF, word embeddings, word2vec, GloVe)